In [14]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [15]:
train_data = datasets.CIFAR10(
    root = "data",   # nereye yükleyeceğiz (data klasörü)
    train = True,
    download = True,    # data zaten yüklü değilse yükle
    transform = transforms.ToTensor(),   # PIL formatında gelen imageleri torch tensor yapar
    target_transform=None   #  you can transform labels as well if needed  (ZATEN NONE gelir)
    )

test_data = datasets.CIFAR10(
    root = "data",   
    train = False,  # test data
    download = True,    
    transform = transforms.ToTensor(),
    )

Files already downloaded and verified
Files already downloaded and verified


In [16]:
image, label = train_data[0]

In [17]:
image

tensor([[[0.2314, 0.1686, 0.1961,  ..., 0.6196, 0.5961, 0.5804],
         [0.0627, 0.0000, 0.0706,  ..., 0.4824, 0.4667, 0.4784],
         [0.0980, 0.0627, 0.1922,  ..., 0.4627, 0.4706, 0.4275],
         ...,
         [0.8157, 0.7882, 0.7765,  ..., 0.6275, 0.2196, 0.2078],
         [0.7059, 0.6784, 0.7294,  ..., 0.7216, 0.3804, 0.3255],
         [0.6941, 0.6588, 0.7020,  ..., 0.8471, 0.5922, 0.4824]],

        [[0.2431, 0.1804, 0.1882,  ..., 0.5176, 0.4902, 0.4863],
         [0.0784, 0.0000, 0.0314,  ..., 0.3451, 0.3255, 0.3412],
         [0.0941, 0.0275, 0.1059,  ..., 0.3294, 0.3294, 0.2863],
         ...,
         [0.6667, 0.6000, 0.6314,  ..., 0.5216, 0.1216, 0.1333],
         [0.5451, 0.4824, 0.5647,  ..., 0.5804, 0.2431, 0.2078],
         [0.5647, 0.5059, 0.5569,  ..., 0.7216, 0.4627, 0.3608]],

        [[0.2471, 0.1765, 0.1686,  ..., 0.4235, 0.4000, 0.4039],
         [0.0784, 0.0000, 0.0000,  ..., 0.2157, 0.1961, 0.2235],
         [0.0824, 0.0000, 0.0314,  ..., 0.1961, 0.1961, 0.

In [18]:
label

6

In [19]:
image.shape

torch.Size([3, 32, 32])

In [20]:
len(train_data), len(test_data)

(50000, 10000)

In [21]:
class_names = train_data.classes
class_names

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [22]:
"""
#image, label = train_data[0]
#plt.imshow(image)
#plt.title(label)
# this will give you an error of 
# TypeError: Invalid shape (3, 32, 32) for image data -- 32,32,3 yapmalıyız
"""

'\n#image, label = train_data[0]\n#plt.imshow(image)\n#plt.title(label)\n# this will give you an error of \n# TypeError: Invalid shape (3, 32, 32) for image data -- 32,32,3 yapmalıyız\n'

In [23]:
"""
image, label = train_data[1]
image = image.permute(1,2,0)
plt.figure(figsize = (2, 2))
plt.imshow(image)
plt.title(class_names[label])
"""

'\nimage, label = train_data[1]\nimage = image.permute(1,2,0)\nplt.figure(figsize = (2, 2))\nplt.imshow(image)\nplt.title(class_names[label])\n'

In [24]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.491, 0.482, 0.446],     # buradaki değerler datasetin ortalaması ve 
        std=[0.247, 0.243, 0.261]       # standart sapmasıdır.
    )
])

CIFAR10 dataset’i 50.000 eğitim görüntüsünden oluşur. Her görüntü: 3 kanal (R, G, B) ve boyut 32×32.

Bu sayılar, CIFAR10 dataset’inin tüm eğitim seti üzerinde hesaplanmış istatistiklerdir.

Yani: • mean (RGB): [0.491, 0.482, 0.446] → CIFAR10 görüntülerinin kırmızı, yeşil ve mavi kanallarının ortalaması. • std (RGB): [0.247, 0.243, 0.261] → Aynı kanalların standart sapması.

CNN’ler, özellikle Adam ve SGD gibi optimizasyon algoritmaları, standartlaştırılmış girişlerle çok daha iyi öğrenir.

Normalize edilmemiş görüntü: • Piksel değerleri 0–255 arasında düzensiz dağılmış • Kanal başına farklı ortalamalar • Modelin ilk katmanı için öğrenme zorlaşır

Normalize edilmiş görüntü: • Tüm görüntünün ortalaması ~ 0 • Dağılımın genişliği ~ 1 • Tüm kanallar daha benzer ölçeğe sahip

Bu: • Modelin hızlı öğrenmesini sağlar • daha stabil loss • Gradientlerin daha “dengeli” akması • Yüksek learning rate’lerin çalışabilmesi • Genel accuracy’nin ciddi şekilde artması

In [25]:
from torch.utils.data import DataLoader

In [26]:
BATCH_SIZE = 32

train_dataloader = DataLoader(
    train_data,
    batch_size = BATCH_SIZE,    # her batchde kaç örnek olacak
    shuffle = True      # karıştırır train datasını
)

test_dataloader = DataLoader(
    test_data,
    batch_size = BATCH_SIZE,    # her batchde kaç örnek olacak
    shuffle = False      # test datasını karıştırmaya gerek yok
)

In [27]:
print(f"Length of train dataloader: {len(train_dataloader)} batches of {BATCH_SIZE}")
print(f"Length of test dataloader: {len(test_dataloader)} batches of {BATCH_SIZE}")

Length of train dataloader: 1563 batches of 32
Length of test dataloader: 313 batches of 32


In [28]:
train_dataloader.dataset[0][0].shape

torch.Size([3, 32, 32])

we have the data. now we need to build the model. before diving into CNNs we will try to do this manually.
even if we are building a very simple model without a cnn architecture, we will still need flatten module.

### Flatten

In [29]:
flatten_model = nn.Flatten()  # example

data = train_dataloader.dataset[0][0]
flattened_data = flatten_model(data)

In [30]:
data.shape

torch.Size([3, 32, 32])

In [31]:
flattened_data.shape  # 32*32 = 1024

torch.Size([3, 1024])

In [32]:
class CIFAR10Classifier(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_shape, out_features=hidden_units),
            nn.Linear(in_features=hidden_units, out_features=output_shape)
        )
    
    def forward(self, x):
        return self.layer_stack(x)

In [33]:
data.shape

torch.Size([3, 32, 32])

In [34]:
3 * 32 * 32  # 3072 pixel -> 3072 feature

3072

In [35]:
model = CIFAR10Classifier(
    input_shape = 3072, 
    hidden_units = 32,  # number of neurons
    output_shape = len(class_names)
)

In [36]:
# model = torch.compile(model)

In [37]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr = 0.001)

In [38]:
def calculate_accuracy(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    return correct / len(y_pred) * 100


In [39]:
torch.manual_seed(42)
epochs = 5

for epoch in range(epochs):
    train_loss = 0

    for batch, (X,y) in enumerate(train_dataloader):
        model.train()

        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 400 == 0:
            print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")

    train_loss /= len(train_dataloader)

    # Test
    test_loss = 0
    test_acc = 0

    model.eval()
    with torch.inference_mode():
        for X,y in test_dataloader:
            test_pred = model(X)

            test_loss += loss_fn(test_pred, y)
            test_acc += calculate_accuracy(y_true = y, y_pred = test_pred.argmax(dim = 1))

        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)

    print(f"\nTrain loss: {train_loss:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%\n")


Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.90657 | Test loss: 1.82741, Test acc: 34.89%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.81437 | Test loss: 1.77471, Test acc: 37.89%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.79097 | Test loss: 1.81676, Test acc: 35.16%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.77505 | Test loss: 1.77128, Test acc: 38.05%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.76622 | Test loss: 1.85962, Test acc: 34.50%



Accuracy rastgeleden daha iyi olsa da istediğimiz kadar iyi sonuç vermiyor.
Bu yüzden CNN ile model kuracagız.

In [40]:
# eval fonksiyonunu oluşturduk. 

def evaluate_model_performance(model, dataloader, loss_fn, accuracy_fn):
    loss, acc = 0, 0

    model.eval()
    with torch.inference_mode():
        for X, y in dataloader:
            y_pred = model(X)
            loss += loss_fn(y_pred, y)
            acc += accuracy_fn(y_true = y, y_pred = y_pred.argmax(dim=1))

        loss /= len(dataloader)
        acc /= len(dataloader)

    return {
        "model_name": model.__class__.__name__,
        "model_loss": loss.item(),
        "model_accuracy": acc
    }

In [41]:
class CIFAR10ClassifierNonLinear(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_shape, out_features=hidden_units),
            nn.LeakyReLU(0.01),
            nn.Linear(in_features=hidden_units, out_features=output_shape)
        )
    
    def forward(self, x):
        return self.layer_stack(x)

In [42]:
torch.manual_seed(31)

model1 = CIFAR10ClassifierNonLinear(3072, 32, len(class_names))

In [43]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model1.parameters(), lr=0.001)

In [44]:
torch.manual_seed(42)
epochs = 5

for epoch in range(epochs):
    train_loss = 0

    for batch, (X,y) in enumerate(train_dataloader):
        model1.train()

        y_pred = model1(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 400 == 0:
            print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")

    train_loss /= len(train_dataloader)

    # Test
    test_loss = 0
    test_acc = 0

    model1.eval()
    with torch.inference_mode():
        for X,y in test_dataloader:
            test_pred = model1(X)

            test_loss += loss_fn(test_pred, y)
            test_acc += calculate_accuracy(y_true = y, y_pred = test_pred.argmax(dim = 1))

        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)

    print(f"\nTrain loss: {train_loss:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%\n")


Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.94586 | Test loss: 1.85863, Test acc: 32.96%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.82289 | Test loss: 1.80268, Test acc: 35.42%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.77667 | Test loss: 1.76893, Test acc: 37.20%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.74506 | Test loss: 1.75644, Test acc: 37.07%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 1.72435 | Test loss: 1.73929, Test acc: 38.32%



In [51]:
class CNNClassifierCIFAR10(nn.Module):
    def __init__(self, hidden_units, input_shape, output_shape):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                in_features = hidden_units * 8 * 8,
                out_features = output_shape
            )
        )


    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))


In [52]:
torch.manual_seed(11)

model2 = CNNClassifierCIFAR10(
    hidden_units=32,
    input_shape=3,  # 3 color channels (rgb)
    output_shape=len(class_names)
)

In [53]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model2.parameters(), lr=0.001)

In [54]:
torch.manual_seed(11)

epochs = 10

for epoch in range(epochs):
    train_loss = 0

    for batch, (X,y) in enumerate(train_dataloader):
        model2.train()

        y_pred = model2(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 400 == 0:
            print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")
    
    train_loss, train_acc = 0, 0

    model2.eval()
    with torch.inference_mode():
        
        for X, y in test_dataloader:
            test_pred = model2(X)
            test_loss += loss_fn(test_pred, y)

            test_acc += calculate_accuracy(y, test_pred.argmax(dim=1))
        

        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)

    
    print(f"\nTrain loss: {train_loss:.3f} | Test loss: {test_loss:.3f}, Test acc: {test_acc:.2f}%\n")
        


Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 0.000 | Test loss: 1.240, Test acc: 56.24%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 0.000 | Test loss: 1.061, Test acc: 62.49%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 0.000 | Test loss: 0.913, Test acc: 68.80%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 0.000 | Test loss: 0.844, Test acc: 70.92%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 0.000 | Test loss: 0.820, Test acc: 72.07%

Looked at 0/50000 samples
Looked at 12800/50000 samples
Looked at 25600/50000 samples
Looked at 38400/50000 samples

Train loss: 0.000 

In [57]:
model_2_results = evaluate_model_performance(model=model2, dataloader=test_dataloader,
    loss_fn=loss_fn, accuracy_fn=calculate_accuracy)

model_1_results = evaluate_model_performance(model=model1, dataloader=test_dataloader,
    loss_fn=loss_fn, accuracy_fn=calculate_accuracy)

model_results = evaluate_model_performance(model=model, dataloader=test_dataloader,
    loss_fn=loss_fn, accuracy_fn=calculate_accuracy)

In [60]:
model_results

{'model_name': 'CIFAR10Classifier',
 'model_loss': 1.8596230745315552,
 'model_accuracy': 34.50479233226837}

In [61]:
model_1_results

{'model_name': 'CIFAR10ClassifierNonLinear',
 'model_loss': 1.7392863035202026,
 'model_accuracy': 38.31869009584665}

In [59]:
model_2_results

{'model_name': 'CNNClassifierCIFAR10',
 'model_loss': 0.7954657673835754,
 'model_accuracy': 73.3326677316294}

In [62]:
def make_predictions(model: torch.nn.Module, data: list):
    """
    data: [img_tensor, img_tensor, ...]  # her biri [C, H, W]
    return: [N, num_classes] olasılık tensörü
    """
    pred_probs = []
    model.eval()

    with torch.inference_mode():
        for sample in data:
            # [C, H, W] -> [1, C, H, W]
            sample = sample.unsqueeze(0)

            # Logits al
            pred_logit = model(sample)    # shape: [1, num_classes]

            # Softmax ile olasılığa çevir
            prob = torch.softmax(pred_logit, dim=1)  # [1, num_classes]

            # Batch boyutunu sıkıştır
            pred_probs.append(prob.squeeze(0))       # [num_classes]

    # Hepsini birleştir → [N, num_classes]
    return torch.stack(pred_probs)

In [63]:
import random
def show_random_predictions(model, dataset, class_names, n=9):
    model.eval()
    
    plt.figure(figsize=(4, 4))

    # random 9 index seç
    indices = random.sample(range(len(dataset)), n)

    with torch.inference_mode():
        for i, idx in enumerate(indices):
            img, true_label = dataset[idx]

            # modele uygun hale getir
            img_input = img.unsqueeze(0)  
            logits = model(img_input)
            pred_label = logits.argmax(dim=1).item()

            # görseli çizmek için permute
            img_show = img.permute(1, 2, 0)

            # doğru mu yanlış mı?
            correct = (pred_label == true_label)
            color = "green" if correct else "red"

            # subplot
            plt.subplot(3, 3, i + 1)
            plt.imshow(img_show)
            plt.axis("off")

            plt.title(
                f"Pred: {class_names[pred_label]}\nTrue: {class_names[true_label]}",
                color=color,
                fontsize=10
            )

    plt.tight_layout()
    plt.show()

In [64]:
show_random_predictions(model, test_data, class_names)

: 

In [ ]:
show_random_predictions(model1, test_data, class_names)

In [ ]:
show_random_predictions(model2, test_data, class_names)